In [1]:
# You only need to run this block once per session to install Gurobi (and other libraries)

%pip install pyomo gurobipy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# v1.1.0 - 9/23/26
# Added constraint to prevent two consecutive work days

import gurobipy as gp
from gurobipy import GRB, Model

import pandas as pd

from datetime import datetime, date as Date
import calendar
from IPython.display import HTML

# =================================
# PARAMETERS
# =================================
YEAR = 2026
MONTH = 1

residents = [
    "A",
    "B",
    "C",
    "D",
    "E",
    "F"
]

vacation_requests = {
    "A": ["2026-01-03", "2026-01-04", "2026-01-17", "2026-01-18"],
    "B": ["2026-01-04", "2026-01-11", "2026-01-24", "2026-01-25"],
    "C": ["2026-01-03", "2026-01-10", "2026-01-17", "2026-01-24"],
    "D": ["2026-01-04", "2026-01-11", "2026-01-18", "2026-01-25"],
    "E": ["2026-01-17", "2026-01-18", "2026-01-31"],
    "F": ["2026-01-24", "2026-01-25", "2026-01-31"],
}

metrics = {
    "max_shifts_assigned":          {"ub": 99, "weight": 1},
    "max_weekend_shifts_assigned":  {"ub": 99, "weight": 1},
    "vacations_denied":             {"ub": 99, "weight": 1},
}


# =================================
# UTILITIES
# =================================
def is_weekend(date_str: str) -> bool:
    return datetime.strptime(date_str, "%Y-%m-%d").weekday() >= 5  # Saturday (5) or Sunday (6)


# =================================
# GENERATE
# =================================
first_date = Date(YEAR, MONTH, 1)
last_day = calendar.monthrange(YEAR, MONTH)[1]
last_date = Date(YEAR, MONTH, last_day)

# Create list of dates, starting from the first date to the last date, in ISO8601 string format
dates : list[Date] = (
    pd.date_range(start=first_date, end=last_date)
    .strftime("%Y-%m-%d")
    .tolist()
)

# =================================
# MODEL PARAMETERS
# =================================
m: Model = Model()
# m.setParam('LogToConsole', 0)


# =================================
# BUILD 
# =================================

# --------- Decision variables ----------
x_rd = m.addVars(len(residents), len(dates), 
                    lb=0, ub=1, 
                    vtype=GRB.BINARY)

# Set variable names
for (r, resident) in enumerate(residents):
    for (d, date) in enumerate(dates):
        x_rd[r, d].VarName = f"x_rd({resident}_{date})"


# --------- Auxiliary variables ----------
alpha_shifts_assigned_r = m.addVars(len(residents),
                                    lb=0, ub=9999, 
                                    vtype=GRB.INTEGER)

alpha_weekend_shifts_assigned_r = m.addVars(len(residents),
                                            lb=0, ub=9999, 
                                            vtype=GRB.INTEGER)

# Set variable names
for (r, resident) in enumerate(residents):
    alpha_shifts_assigned_r[r].VarName = f"alpha_shifts_assigned_r({resident})"
    alpha_weekend_shifts_assigned_r[r].VarName = f"alpha_weekend_shifts_assigned_r({resident})"


# --------- Metric variables ----------
for metric_name, metric in metrics.items():
    metric["var"] = m.addVar(lb=0, ub=metric["ub"], 
                        vtype=GRB.INTEGER)
    metric["var"].VarName = metric_name


# --------- Constraints ----------
# One res for each date
for (d, date) in enumerate(dates):
    m.addConstr(gp.quicksum(x_rd[r, d] for (r, resident) in enumerate(residents)) == 1,
                name=f"single_shift_d({date})")

# Resident coverage ub of 7 across the entire month
for (r, resident) in enumerate(residents):
    m.addConstr(gp.quicksum(x_rd[r, d] for (d, date) in enumerate(dates)) <= 7,
                name=f"coverage_ub_r({resident})")
    
# Each resident cannot work two consecutive days
for (r, resident) in enumerate(residents):
    for (d, date) in enumerate(dates[:-1]):
        m.addConstr(x_rd[r, d] + x_rd[r, d + 1] <= 1,
                    name=f"prohibit_consecutive_days_rd({resident}_{date})")


# --------- Linking Constraints ----------
# Define alpha_shifts_assigned_r as number of shifts assigned to resident r in the entire month
for (r, resident) in enumerate(residents):
    m.addConstr(
        alpha_shifts_assigned_r[r] >= gp.quicksum(
            x_rd[r, d]
            for d, shift_date in enumerate(dates)
        ),
        name=f"alpha_shifts_assigned_r({resident})"
    )

# Define alpha_weekend_shifts_assigned_r as number of weekend shifts assigned to resident r in the entire month
for (r, resident) in enumerate(residents):
    m.addConstr(
        alpha_weekend_shifts_assigned_r[r] >= gp.quicksum(
            x_rd[r, d]
            for d, shift_date in enumerate(dates)
            if is_weekend(shift_date)
        ),
        name=f"alpha_weekend_shifts_assigned_r({resident})"
    )


# --------- Metric Constraints ----------
# Define max_shifts_assigned
for (r, resident) in enumerate(residents):
    m.addConstr(
        metrics["max_shifts_assigned"]["var"] >= alpha_shifts_assigned_r[r],
        name=f"max_shifts_assigned_r({resident})"
    )

# Define max_weekend_shifts_assigned
for (r, resident) in enumerate(residents):
    m.addConstr(
        metrics["max_weekend_shifts_assigned"]["var"] >= alpha_weekend_shifts_assigned_r[r],
        name=f"max_weekend_shifts_assigned_r({resident})"
    )

# Define vacations_denied
m.addConstr(
    metrics["vacations_denied"]["var"] >= gp.quicksum(
        x_rd[r, d]
        for r, resident in enumerate(residents)
        for d, shift_date in enumerate(dates)
        if shift_date in vacation_requests[resident]
    ),
    name="metric_vacations_denied"
)



# --------- Objective function ----------
# Set objective function to weighted sum of all metrics
m.setObjective(
    gp.quicksum(metric["weight"] * metric["var"] for metric in metrics.values()),
    GRB.MINIMIZE
)


# =================================
# SOLVE
# =================================
m.optimize()

# Only show reports if model is feasible
if m.Status != GRB.INFEASIBLE:
    tolerance = 0.5
    # Extract solution variable as boolean
    x_rd_sol = [[x_rd[i, j].X > tolerance for j in range(len(dates))]
            for i in range(len(residents))]

    # =================================
    # REPORT
    # =================================

    # ---------- Deubgging ----------
    current = datetime.now().strftime("%m-%d-%Y-%H-%M-%S")
    m.write(f"output/{current}-model.lp")


    # ---------- Table ----------
    table_rows = []

    for r, resident in enumerate(residents):
        total_shifts = sum(
            x_rd_sol[r][d]
            for d in range(len(dates))
        )

        weekend_shifts = sum(
            x_rd_sol[r][d]
            for d, shift_date in enumerate(dates)
            if is_weekend(shift_date)
        )

        denied_requests = sum(
            x_rd_sol[r][d]
            for d, shift_date in enumerate(dates)
            if shift_date in vacation_requests[resident]
        )

        requested_vacations = len(vacation_requests[resident])

        table_rows.append({
            "Resident": resident,
            "Total Shifts": total_shifts,
            "Weekend Shifts": weekend_shifts,
            "Vacation Requests": requested_vacations,
            "Requests Denied": denied_requests,
        })

    table_rows_df = pd.DataFrame(table_rows)
    display(table_rows_df)

    # ---------- Calendar ----------
    class Schedule(calendar.HTMLCalendar):
        def __init__(self, event_names, dates, occurs):
            # initialize calendar.HTMLCalendar to start weeks on Sunday
            super().__init__(calendar.SUNDAY)

            self.assignments = {}
            for r in range(len(residents)):
                for d in range(len(dates)):
                    if (x_rd_sol[r][d]):
                        self.assignments[dates[d]] = residents[r]
            

        def formatday(self, day, weekday):
            if day == 0:
                # empty table cell
                return '<td></td>'

            # render cell with day and assignment
            event_date = Date(
                self.current_year,
                self.current_month,
                day
            ).strftime("%Y-%m-%d")
            event = self.assignments.get(event_date, "")

            event_html = (f'<div>{event}</div>') if event else ""
            return (f'<td style="border:1px solid #808080; width:80px; height:60px; vertical-align:top">'
                    f'<strong>{day}</strong>'
                    f'{event_html}'
                    f'</td>')

        def formatmonth(self, theyear, themonth, withyear=True):
            self.current_year = theyear
            self.current_month = themonth
            return super().formatmonth(theyear, themonth, withyear)

    schedule = Schedule(residents, dates, x_rd_sol)
    display(HTML(schedule.formatmonth(YEAR, MONTH)))

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 7 265U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 242 rows, 201 columns and 1031 nonzeros (Min)
Model fingerprint: 0x8542163e
Model has 3 linear objective coefficients
Variable types: 0 continuous, 201 integer (186 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+04]
  RHS range        [1e+00, 7e+00]

Found heuristic solution: objective 12.0000000
Presolve removed 13 rows and 13 columns
Presolve time: 0.00s
Presolved: 229 rows, 188 columns, 984 nonzeros
Variable types: 0 continuous, 188 integer (186 binary)
Found heuristic solution: objective 11.0000000

Root relaxation: objective 6.666667e+00, 100 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds  

,Resident,Total Shifts,Weekend Shifts,Vacation Requests,Requests Denied
0,A,6,2,4,0
1,B,5,1,4,0
2,C,5,1,4,0
3,D,6,2,4,0
4,E,4,2,3,0
5,F,5,1,3,0
